In [1]:
import sys
import os
import geopandas as gpd
import pandas as pd
import pandapower as pp

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from power_flow import build_pandapower_model

load c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\ortools\.libs\zlib1.dll...
load c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\ortools\.libs\abseil_dll.dll...
load c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\ortools\.libs\utf8_validity.dll...
load c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\ortools\.libs\re2.dll...
load c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\ortools\.libs\libprotobuf.dll...
load c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\ortools\.libs\highs.dll...
load c:\Users\mye500\AppData\Local\anaconda3\envs\py311\Lib\site-packages\ortools\.libs\ortools.dll...


In [2]:
# === Read power network data ===
lines_gdf = gpd.read_file('../outputs/table_lines_200m_update.gpkg')
nodes_gdf = gpd.read_file('../outputs/table_nodes_200m_update.gpkg')
gens_gdf = gpd.read_file('../outputs/plant_update.gpkg')
loads_gdf = gpd.read_file('../outputs/landuse_sites_gdf_add_bus.gpkg') # updated version data

In [3]:
net = build_pandapower_model(nodes_gdf, lines_gdf, gens_gdf, loads_gdf)

In [ ]:
net.load.head()

In [ ]:
import copy
import pandapower as pp
import pandapower.topology as top
import numpy as np
import pandas as pd
import warnings

def simulate_percolation_with_powerflow(build_pandapower_model, num_iterations=30, remove_fractions=None):
    """
    每轮 deepcopy 一个 base_net，执行结构断连 + 潮流分析

    参数：
        build_model_func: 构建 pandapower 网络的函数
        num_iterations: 每个比例下模拟次数
        remove_fractions: List[float]，默认每 5%

    返回：
        DataFrame，包含渗流与潮流分析结果
    """
    if remove_fractions is None:
        remove_fractions = np.linspace(0, 0.4, 9)  # 默认每5%

    base_net = build_pandapower_model(nodes_gdf, lines_gdf, gens_gdf, loads_gdf)  # 只构建一次网络
    all_buses = base_net.bus.index.tolist()
    results = []

    for frac in remove_fractions:
        for i in range(num_iterations):
            net = copy.deepcopy(base_net)
            net.bus["in_service"] = True

            n_remove = int(frac * len(all_buses))
            failed_buses = np.random.choice(all_buses, size=n_remove, replace=False)
            net.bus.loc[failed_buses, "in_service"] = False

            try:
                G = top.create_nxgraph(net, include_switches=True)
                comps = list(top.connected_components(G))
                giant_size = max((len(c) for c in comps), default=0)
            except Exception:
                giant_size = 0

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                try:
                    pp.runpp(net, init="auto", calculate_voltage_angles=True)
                    success = True
                except:
                    success = False

            if success:
                voltages = net.res_bus.vm_pu[net.bus.in_service]
                under = sum(voltages < 0.95)
                over = sum(voltages > 1.05)
                voltage_violations = under + over

                load_bus_status = net.bus.loc[net.load.bus.values, "in_service"].values
                unserved_loads = sum(~load_bus_status)

                total_load = net.load.p_mw.sum()
                total_supplied = net.res_load.p_mw.sum()
            else:
                voltage_violations = np.nan
                unserved_loads = np.nan
                total_supplied = np.nan
                total_load = net.load.p_mw.sum()

            results.append({
                "removal_fraction": frac,
                "iteration": i,
                "giant_component_fraction": giant_size / len(all_buses),
                "powerflow_success": success,
                "voltage_violations": voltage_violations,
                "unserved_loads": unserved_loads,
                "total_load": total_load,
                "total_supplied": total_supplied
            })

    return pd.DataFrame(results)


In [ ]:
df = simulate_percolation_with_powerflow(build_pandapower_model, num_iterations=2)

In [7]:
def simulate_percolation_with_powerflow_and_loadstatus(build_pandapower_model, num_iterations=30, remove_fractions=None):
    """
    执行渗流 + 潮流分析，并记录每轮每个负载是否成功供电。
    返回两个结果：
    - overall_df: 每轮网络级指标
    - load_status_df: 每个负载是否成功供电的逐轮记录
    """
    if remove_fractions is None:
        remove_fractions = np.linspace(0, 0.2, 5)

    base_net = build_pandapower_model(nodes_gdf, lines_gdf, gens_gdf, loads_gdf)
    all_buses = base_net.bus.index.tolist()
    results = []
    load_status_records = []

    for frac in remove_fractions:
        for i in range(num_iterations):
            net = copy.deepcopy(base_net)
            net.bus["in_service"] = True

            n_remove = int(frac * len(all_buses))
            failed_buses = np.random.choice(all_buses, size=n_remove, replace=False)  # Monte-carlo??
            net.bus.loc[failed_buses, "in_service"] = False

            try:
                G = top.create_nxgraph(net, include_switches=True)
                comps = list(top.connected_components(G))
                giant_size = max((len(c) for c in comps), default=0)
            except Exception:
                giant_size = 0

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                try:
                    pp.runpp(net, init="auto", calculate_voltage_angles=True)
                    success = True
                except:
                    success = False
                    
            if success:
                voltages = net.res_bus.vm_pu[net.bus.in_service]
                under = sum(voltages < 0.95)
                over = sum(voltages > 1.05)
                voltage_violations = under + over

                total_load = net.load.p_mw.sum()
                total_supplied = net.res_load.p_mw.sum()

                # 每个负载的供电状态记录
                for idx, row in net.load.iterrows():
                    load_bus = row["bus"]
                    is_served = (
                        net.bus.at[load_bus, "in_service"]
                        and not np.isnan(net.res_load.at[idx, "p_mw"])
                        and net.res_load.at[idx, "p_mw"] > 0
                    )
                    load_status_records.append({
                        "iteration": i,
                        "removal_fraction": frac,
                        "load_id": idx,
                        "bus": load_bus,
                        "served": is_served
                    })
            else:
                voltage_violations = np.nan
                total_load = net.load.p_mw.sum()
                total_supplied = np.nan

                # 无法运行潮流时，所有负载都设为未供电
                for idx, row in net.load.iterrows():
                    load_status_records.append({
                        "iteration": i,
                        "removal_fraction": frac,
                        "load_id": row["name"],
                        "bus": row["bus"],
                        "served": False
                    })

            results.append({
                "removal_fraction": frac,
                "iteration": i,
                "giant_component_fraction": giant_size / len(all_buses),
                "powerflow_success": success,
                "voltage_violations": voltage_violations,
                "total_load": total_load,
                "total_supplied": total_supplied
            })

    return pd.DataFrame(results), pd.DataFrame(load_status_records)


In [ ]:
df_new, load_status_record = simulate_percolation_with_powerflow_and_loadstatus(build_pandapower_model, 2)

In [ ]:
load_status_record

In [ ]:
# 可视化
import matplotlib.pyplot as plt
df_mean = df_new.groupby("removal_fraction")["giant_component_fraction"].mean()
df_std = df_new.groupby("removal_fraction")["giant_component_fraction"].std()

plt.figure(figsize=(8,5))
plt.plot(df_mean.index, df_mean, label="Mean Giant Component")
plt.fill_between(df_mean.index, df_mean - df_std, df_mean + df_std, alpha=0.3, label="±1 std")
plt.xlabel("Fraction of buses removed")
plt.ylabel("Giant component fraction")
plt.title("Percolation analysis (pandapower network)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def prepare_loads_map(load_status_df, loads_gdf, target_iteration=None, target_fraction=None):
    """
    将 load_status_df 统计结果合并到 loads_gdf 中，并绘制地图
    - 若指定 iteration 和 removal_fraction，则显示该轮供电状态
    - 否则计算每个负载被供电的次数比例（平均情况）
    """
    # 复制 loads_gdf，避免修改原始数据
    loads_map_gdf = loads_gdf.copy()

    # 重命名 load_id 为 osmid 对齐
    status_df = load_status_df.rename(columns={"load_id": "osmid"})

    if target_iteration is not None and target_fraction is not None:
        # 提取指定轮次和比例的状态
        filtered = status_df[
            (status_df["iteration"] == target_iteration) &
            (status_df["removal_fraction"] == target_fraction)
        ]
        served_map = dict(zip(filtered.osmid, filtered.served))
        loads_map_gdf["served"] = loads_map_gdf["osmid"].map(served_map).fillna(False)

    else:
        # 计算每个负载被供电的次数比例
        avg_status = (
            status_df.groupby("osmid")["served"]
            .mean()
            .reset_index()
            .rename(columns={"served": "served_ratio"})
        )
        loads_map_gdf = loads_map_gdf.merge(avg_status, on="osmid", how="left")
        loads_map_gdf["served_ratio"] = loads_map_gdf["served_ratio"].fillna(0)

    return loads_map_gdf


In [ ]:
# 平均供电比例地图（推荐用来判断脆弱性）
loads_result = prepare_loads_map(load_status_record, loads_gdf)

# 某一轮结果（例如第3轮，20%节点失效）
loads_result = prepare_loads_map(load_status_record, loads_gdf, target_iteration=0, target_fraction=0.2)


In [ ]:
loads_result

In [ ]:
import matplotlib.pyplot as plt

def plot_loads_binary_map(loads_result_gdf, title="Load Supply Map", output_path=None):
    """
    根据 'served' 字段绘制红绿二值供电地图
    红色 = 未供电，绿色 = 已供电
    """
    fig, ax = plt.subplots(figsize=(10, 10))

    # 供电与否分组绘制
    served = loads_result_gdf[loads_result_gdf["served"] == True]
    not_served = loads_result_gdf[loads_result_gdf["served"] == False]

    served.plot(ax=ax, color='green', markersize=10, label='Served')
    not_served.plot(ax=ax, color='red', markersize=10, label='Not Served')

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=300)
    plt.show()


In [ ]:
# 1. 准备数据（选择某一轮）
loads_map = prepare_loads_map(load_status_df, loads_gdf, target_iteration=0, target_fraction=0.2)

# 2. 绘图
plot_loads_binary_map(loads_map, title="Load Supply Status (Iter 0, 20% Failure)")
